##E1.1 Variables, tipos y operaciones — SETUP

In [0]:
%py

zona="Tucuman"
precio=121575.99
moneda="ARS"
metros=65
ambientes=3
tiene_cochera=True


print(f"Tipo dato var zona: {type(zona)}")
print(f"Tipo dato var precio: {type(precio)}")
print(f"Tipo dato var moneda: {type(moneda)}")
print(f"Tipo dato var metros: {type(metros)}")
print(f"Tipo dato var ambientes: {type(ambientes)}")
print(f"Tipo dato var tiene_cochera: {type(tiene_cochera)}")

print("\n")

precio_m2=precio/metros

print(f"El precio por m2 es de $ {round(precio_m2,2)}")

print("\n")

print(f"ZONA {zona.upper()}")
print(f"ZONA {zona.lower()}")
print("\n")
# -- casting
metros_float=float(metros)
print(f"metros_float: {metros_float}")
print(f"metros_float: {type(metros_float)}")

##E1.2 f-strings para reporte — SETUP

In [0]:
%py

zonas=[
    {"partido":"SM TUC","moneda":"ARS","precio":333444.0},
    {"partido":"YERBA BUENA","moneda":"ARS", "precio":555666.0},
    {"partido":"LULES","moneda":"ARS","precio":111222.0}
]

print(f"{"Partido":<15} {"Moneda"} {"Precio":>12}")

for zona in zonas:
    print(f"{zona["partido"]:<15} {zona["moneda"]} {zona["precio"]:>15.2f}")


##E1.3 list + set + dict — GUIDED

In [0]:
%py
lista_precios=[100,200,500,600,900,300]

print(f"Lista de precios length: {len(lista_precios)}")
print(f"El precio maximo es: {max(lista_precios)}")
print(f"El precio minimo es: {min(lista_precios)}")
print(f"El primer precio es: {lista_precios[0]}")
print(f"El ultimo precio es: {lista_precios[-1]}")

lista_ordenada=sorted(lista_precios)
print(f"Lista original: {lista_precios}")
print(f"Lista ordenada: {lista_ordenada}")

lista_precios.append(1250)

print(f"Lista precios modificada: {lista_precios}")


In [0]:
%py
lista_zonas=["sm_tuc","yerba_buena","lules","bda_rio_sali","sm_tuc","la_cocha"]

set_zonas=set(lista_zonas)
print(f"List de zonas original {lista_zonas}")
print(f"Set de zonas sin duplicados {sorted(set_zonas)}")

zonas_1={"palermo","belgrano","caballito"}
zonas_2={"san_telmo","belgrano","flores"}

print(f"\nUnion: {zonas_1.union(zonas_2)}")
print(f"Interseccion: {zonas_1.intersection(zonas_2)}")
print(f"Diferencia en z1 y z2: {zonas_1.difference(zonas_2)}")
print(f"Diferencia en z2 y z1: {zonas_2.difference(zonas_1)}")

In [0]:
%py

propiedad={
    "partido":"sm_tucuman",
    "precio":250000.0,
    "moneda":"USD",
    "ambientes":3,
    "metros":65.0
}


print("##PROPIEDAD##")

print(f"Partido: {propiedad["partido"]}")
print(f"Precio: {propiedad["precio"]}")
print(f"Moneda: {propiedad["moneda"]}")
print(f"Ambientes: {propiedad["ambientes"]}")
print(f"Metros: {propiedad["metros"]}")

print("\n")
print("##KEYS")
print(f"{propiedad.keys()}")
print("##VALUES##")
print(f"{propiedad.values()}")
print("##ITEMS##")
print(f"{propiedad.items()}")

propiedad["precio_m2"]=propiedad["precio"]/propiedad["metros"]

print("\n")
print("##PROPIEDAD##")
print(f"Precio por m2: {propiedad["precio_m2"]:.2f}")



##E1.4 for loop + función — GUIDED

In [0]:
%py
def calcula_m2(precio,metros):
    if metros==0:
        return 0
    else:
        return precio/metros



propiedades=[
    {"partido":"SM TUC","moneda":"ARS","precio":333444.0,"metros_2":65.0},
    {"partido":"YERBA BUENA","moneda":"ARS", "precio":555666.0,"metros_2":0},
    {"partido":"LULES","moneda":"ARS","precio":111222.0,"metros_2":78},
    {"partido":"SAN PABLO","moneda":"ARS","precio":999777.0,"metros_2":178}
]

print(f"{"ITEM":>5} {"PARTIDO":<15} {"MONEDA":<10} {"M2":>10} {"PRECIO":>15} {"PRECIO X M2":>15}\n")

for i, prop in enumerate(propiedades):
    print(f"{i+1:>5} {prop["partido"]:<15} {prop["moneda"]:<10} {prop["metros_2"]:>10} {prop["precio"]:>15} {calcula_m2(prop["precio"],prop["metros_2"]):>15.2f}")
    print(f"{"-"*75}")

##E1.5 Python → spark.sql() con f-strings — GUIDED

In [0]:
%py

zonas=["ezeiza","pilar","tigre","moreno"]

zonas=sorted(zonas)

print(f"{"ZONA":<15} {"CANTIDAD PROPIEDADES":>15}")
print(f"{"-"*50}")

for zona in zonas:
    n=spark.sql(f"""select count(*) as c from bootcamp.silver.propiedades where partido='{zona}'"""                
                ).collect()[0].c
    print(f"{zona.upper():<15} {n:>15}")
    print(f"{"-"*50}")

##E1.6 SparkSession y Lazy Evaluation — INDEPENDENT

In [0]:
%py

version=spark.version

cores=int(spark.conf.get("spark.default.parallelism", "200"))

print(f"VERSION: {version}")
print(f"CORES: {cores}")

In [0]:
%py

df_props=spark.sql(f"""
                   SELECT
                       propiedad_id,
                       partido,
                       precio,
                       ambientes
                   FROM
                       bootcamp.silver.propiedades
                    WHERE tipo_operacion='venta' AND moneda='USD'
                   """)

df_props.explain(True)

In [0]:
%py
from pyspark.sql.functions import col


df_propiedades=spark.table("bootcamp.silver.propiedades")


df_filtrado=(df_propiedades
             .filter(col("tipo_operacion")=='venta')
             .filter(col("moneda")=='USD')
             .select("propiedad_id","partido","precio","ambientes")
             )

df_filtrado.explain(True)   



## E1.7 Decision Framework

In [0]:
%sql

SELECT
    partido,
    COUNT(*) as total_propiedades,
    AVG(precio) as precio_promedio
FROM bootcamp.silver.propiedades
WHERE tipo_operacion='venta'
GROUP BY partido
ORDER BY total_propiedades DESC
LIMIT 5;

In [0]:
%py

df_propiedades=spark.sql(f"""
                         SELECT
                            partido,
                            COUNT(*) as total_propiedades,
                            AVG(precio) as precio_promedio
                         FROM bootcamp.silver.propiedades
                         WHERE tipo_operacion='venta'
                         GROUP BY partido
                         ORDER BY total_propiedades DESC
                         LIMIT 5
                         """)


#df_propiedades.explain(True)
   
df_propiedades.show()


In [0]:
%py
from pyspark.sql.functions import col, count, avg
tabla_propiedades=spark.table("bootcamp.silver.propiedades")

df_result=tabla_propiedades.filter(col("tipo_operacion")=='venta').groupBy("partido").agg(count("*").alias("total_propiedades"),avg("precio").alias("precio_promedio")).orderBy(col("total_propiedades").desc()).limit(5)

#df_result.explain(True)
df_result.show()